# ML5 — PCA vs Truncated SVD on the same sparse matrix

**Project:** DS 5020 Mini-Project "Customers Like You".

Inputs (read-only): `A_log.npz`, `customer_index.parquet`, `product_index.parquet`, `test_targets.parquet`.
Code reused unchanged: `src/svd.py` (fit, energy / error helpers, component inspection), `src/recommend.py`
(product vectors, scoring, best-seller baseline) and `src/evaluate.py` (evaluation set, metrics).

Both methods use `n_components = 50` and the **same** recommender pipeline afterwards: product vectors are
`components_.T` with unit-length rows, scores are `A_log[row] @ P @ P.T`, already-bought products are excluded, top 10.

| | Truncated SVD | PCA |
|---|---|---|
| estimator | `TruncatedSVD(50, algorithm="randomized", n_iter=7, random_state=42)` | `PCA(50, svd_solver="arpack", random_state=42)` (fallback `covariance_eigh`) |
| input | sparse `A_log` as is | the **same** sparse `A_log`; sklearn centres each column implicitly |
| scoring rows | `A_log` rows, uncentred | `A_log` rows, **uncentred** (the PCA mean is *not* subtracted at scoring time) |

Outputs: `reports/ml5_comparison.csv` and `reports/ml5_comparison.md` (brief table, *Measure | PCA | Truncated SVD*).
The row "What component 1 represents, in words" is left blank on purpose.

In [1]:
# Environment guard: this notebook must run in the "giftlane" conda env (pandas 3.0.6).
# It fails fast with "Wrong Python environment. Run: conda activate giftlane" otherwise.
import sys
from pathlib import Path

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.env_check import check_env

print("environment OK:", check_env())

environment OK: /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python


## 0. Setup and inputs

In [2]:
import time
import tracemalloc

import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
from sklearn.decomposition import PCA, TruncatedSVD

from src.svd import SEED, N_ITER, load_inputs, total_energy, energy_kept, rel_error, factor_size_mb, fit_svd, top_products_for_component
from src.recommend import Recommender, product_vectors, recommend_for_customers
from src.evaluate import TOP_N, build_eval_set, evaluate_baseline, per_customer_metrics

ROOT = _ROOT
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

K = 50
A_log, product_index, customer_index = load_inputs(ROOT)
targets = pd.read_parquet(PROCESSED / "test_targets.parquet")
ev = build_eval_set(targets, customer_index, product_index)
energy = total_energy(A_log)
n, m = A_log.shape

print("python  :", sys.executable)
print(f"sklearn : {sklearn.__version__} | numpy : {np.__version__} | pandas : {pd.__version__} | scipy : {__import__('scipy').__version__}")
print(f"A_log   : {n:,} x {m:,}, nnz {A_log.nnz:,}, ||A||_F^2 = {energy:,.2f}")
print(f"eval set: {ev.n_eval:,} customers, {int(ev.n_targets.sum()):,} target pairs ({int(ev.n_targets_in_matrix.sum()):,} in matrix), ceiling {ev.ceiling_pct:.2f}%")
assert (n, m) == (5471, 3687) and A_log.nnz == 413706, "frozen matrix changed?"
assert ev.n_eval == 1931

python  : /opt/homebrew/Caskroom/miniforge/base/envs/giftlane/bin/python
sklearn : 1.9.1 | numpy : 2.4.6 | pandas : 3.0.6 | scipy : 1.17.1
A_log   : 5,471 x 3,687, nnz 413,706, ||A||_F^2 = 2,389,340.13
eval set: 1,931 customers, 49,647 target pairs (44,922 in matrix), ceiling 99.07%


## 1. Fit both models

Timing: 3 fresh fits each, median wall-clock, **without** `tracemalloc` running (it slows allocations down).
Peak memory: one extra fit of each with `tracemalloc` on; it tracks allocations made through Python's allocator
(numpy arrays included) but not Fortran/BLAS workspaces inside ARPACK or LAPACK, so it is a lower bound.

In [3]:
def fit_pca(A, k=K, seed=SEED):
    """PCA on the sparse matrix. arpack first (implicit centring); covariance_eigh as fallback."""
    for solver in ("arpack", "covariance_eigh"):
        try:
            model = PCA(n_components=k, svd_solver=solver, random_state=seed)
            t0 = time.perf_counter()
            model.fit(A)
            return model, time.perf_counter() - t0, solver
        except Exception as exc:  # noqa: BLE001 - report and try the next solver
            print(f"PCA svd_solver={solver!r} failed: {type(exc).__name__}: {exc}")
    raise RuntimeError("no PCA solver accepted the sparse input")


def fit_tsvd(A, k=K, seed=SEED):
    model, secs = fit_svd(A, k, seed=seed, n_iter=N_ITER)
    return model, secs, "randomized"


def timed_fits(fit_fn, A, repeats=3):
    models, secs, solvers = [], [], []
    for _ in range(repeats):
        mdl, s, solver = fit_fn(A)
        models.append(mdl); secs.append(s); solvers.append(solver)
    assert len(set(solvers)) == 1
    return models[-1], secs, solvers[0]


def peak_memory_mb(fit_fn, A):
    tracemalloc.start()
    tracemalloc.reset_peak()
    fit_fn(A)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1e6


svd, svd_secs, svd_solver = timed_fits(fit_tsvd, A_log)
pca, pca_secs, pca_solver = timed_fits(fit_pca, A_log)
svd_peak = peak_memory_mb(fit_tsvd, A_log)
pca_peak = peak_memory_mb(fit_pca, A_log)

print(f"Truncated SVD : solver {svd_solver:<16} fit times {np.round(svd_secs, 3).tolist()} s -> median {np.median(svd_secs):.3f} s | peak traced memory {svd_peak:.1f} MB")
print(f"PCA           : solver {pca_solver:<16} fit times {np.round(pca_secs, 3).tolist()} s -> median {np.median(pca_secs):.3f} s | peak traced memory {pca_peak:.1f} MB")
print(f"PCA solver used: {pca_solver}" + ("" if pca_solver == "arpack" else "  (arpack failed on sparse input, see message above)"))

# both component matrices are orthonormal (rows) - required for the cosine / projection scoring to mean the same thing
for name, mdl in (("SVD", svd), ("PCA", pca)):
    G = mdl.components_ @ mdl.components_.T
    assert np.allclose(G, np.eye(K), atol=1e-8), f"{name} components not orthonormal"
print("components_ rows orthonormal for both models: OK")

Truncated SVD : solver randomized       fit times [0.104, 0.102, 0.101] s -> median 0.102 s | peak traced memory 9.7 MB
PCA           : solver arpack           fit times [0.099, 0.095, 0.095] s -> median 0.095 s | peak traced memory 8.3 MB
PCA solver used: arpack
components_ rows orthonormal for both models: OK


## 2. Stored factors and variance / energy kept

Stored factors = customer scores (`transform(A_log)`, n x k) + product components (m x k), float64.
PCA additionally needs the column-mean vector `mean_` (m floats) to centre new data; it is listed separately.

In [4]:
U_svd = svd.transform(A_log)        # = U * Sigma, (n, k)
Z_pca = pca.transform(A_log)        # centred scores, (n, k)
assert U_svd.shape == Z_pca.shape == (n, K)

factors_mb = factor_size_mb(n, m, K)     # identical formula for both: (n + m) * k * 8 bytes
pca_mean_mb = m * 8 / 1e6
svd_rel_err = rel_error(svd.singular_values_, energy)
svd_energy_pct = 100 * energy_kept(svd.singular_values_, energy)
pca_evr_pct = 100 * float(pca.explained_variance_ratio_.sum())

print(f"stored factors (float64, scores + components): {factors_mb:.2f} MB for each method  [{n:,}+{m:,} rows x {K} x 8 B]")
print(f"  PCA also stores mean_ ({m:,} floats): +{pca_mean_mb:.3f} MB")
print(f"SVD : relative reconstruction error sqrt(1 - sum(sigma^2)/||A||_F^2) = {svd_rel_err:.4f} ; energy kept = {svd_energy_pct:.2f}%")
print(f"PCA : explained_variance_ratio_.sum() = {pca_evr_pct:.2f}%  (share of total *centred* variance, not of ||A||_F^2)")
print(f"      leading explained-variance ratios: {np.round(100 * pca.explained_variance_ratio_[:5], 2).tolist()} %")
print(f"      leading SVD energy shares        : {np.round(100 * svd.singular_values_[:5]**2 / energy, 2).tolist()} %")

stored factors (float64, scores + components): 3.66 MB for each method  [5,471+3,687 rows x 50 x 8 B]
  PCA also stores mean_ (3,687 floats): +0.029 MB
SVD : relative reconstruction error sqrt(1 - sum(sigma^2)/||A||_F^2) = 0.7892 ; energy kept = 37.71%
PCA : explained_variance_ratio_.sum() = 34.99%  (share of total *centred* variance, not of ||A||_F^2)
      leading explained-variance ratios: [9.3, 2.23, 1.81, 1.19, 1.15] %
      leading SVD energy shares        : [13.02, 2.17, 1.74, 1.14, 1.12] %


## 3. Recommendation quality on the 1,931 evaluation customers

Same pipeline for both models, via `src.recommend` / `src.evaluate`:
`P = normalize(components_.T)` (weighting `"V"`), `scores = A_log[rows] @ P @ P.T`, already-bought masked, top 10.

**PCA note.** The customer rows fed to the scorer are the raw `A_log` rows; the column means that PCA subtracted
during fitting are *not* subtracted here. The projection `P @ P.T` is the same orthogonal projection either way,
but the score vector differs from PCA's own `transform` by the constant term `(-mean) @ P @ P.T`, which is identical
for every customer and so only shifts all products' scores by a per-product constant.

In [5]:
def as_recommender(model, fit_secs, label):
    P, zero_norm = product_vectors(model, "V")          # components_.T, unit-norm rows (src.recommend)
    return Recommender(svd=model, k=K, weighting="V", P=P, A=A_log, product_index=product_index,
                       zero_norm_cols=zero_norm, fit_seconds=fit_secs, meta={"method": label})


rec_svd = as_recommender(svd, float(np.median(svd_secs)), "Truncated SVD")
rec_pca = as_recommender(pca, float(np.median(pca_secs)), "PCA")
print("zero-norm product vectors: SVD", len(rec_svd.zero_norm_cols), "| PCA", len(rec_pca.zero_norm_cols))

quality = {}
for label, rec in (("Truncated SVD", rec_svd), ("PCA", rec_pca)):
    recs, _, secs = recommend_for_customers(rec, ev.rows, n=TOP_N)
    pc = per_customer_metrics(recs, ev, TOP_N)
    quality[label] = {**pc.summary(ev.reachable), "score_eval_s": secs, "total_hits": int(pc.hits.sum())}

base_row, base_pc, _ = evaluate_baseline(rec_svd, ev, TOP_N)   # depends only on product_index.n_customers and A_log

print(f"{'method':<22}{'hit@10':>9}{'recall@10':>12}{'precision@10':>15}{'total hits':>12}")
for label in ("PCA", "Truncated SVD"):
    q = quality[label]
    print(f"{label:<22}{q['hit@10']:8.2f}%{q['recall@10']:11.2f}%{q['precision@10']:14.2f}%{q['total_hits']:12,d}")
print(f"{'best-seller baseline':<22}{base_row['hit@10']:8.2f}%{base_row['recall@10']:11.2f}%{base_row['precision@10']:14.2f}%{int(base_pc.hits.sum()):12,d}")
print("PCA scoring: customer rows NOT centred (raw A_log rows).")

# the SVD numbers must reproduce ML4's "SVD k=50 V log" row (same seed, same code path)
ml4 = pd.read_csv(REPORTS / "ml4_results.csv").set_index("config").loc["SVD k=50 V log"]
for mtr in ("hit@10", "recall@10", "precision@10"):
    assert abs(ml4[mtr] - quality["Truncated SVD"][mtr]) < 1e-9, mtr
print("SVD k=50 V log reproduces reports/ml4_results.csv exactly: OK")

zero-norm product vectors: SVD 0 | PCA 0
method                   hit@10   recall@10   precision@10  total hits
PCA                      20.66%       2.26%          3.09%         597
Truncated SVD            21.44%       2.35%          3.15%         609
best-seller baseline     26.77%       1.74%          3.62%         699
PCA scoring: customer rows NOT centred (raw A_log rows).
SVD k=50 V log reproduces reports/ml4_results.csv exactly: OK


## 4. Component 1: top products, sign pattern, relation to column means

Sign convention: the sign of a singular vector / principal axis is arbitrary (flipping `v` and `u` together changes
nothing), so "all negative" and "all positive" are the same pattern; what matters is whether the weights are
all one sign or mixed.

In [6]:
def sign_pattern(v, top_idx):
    pos, neg = int((v > 0).sum()), int((v < 0).sum())
    top_signs = np.sign(v[top_idx])
    if pos == 0 or neg == 0:
        full = "all one sign (" + ("positive" if neg == 0 else "negative") + ")"
    else:
        full = f"mixed: {pos:,} positive / {neg:,} negative of {len(v):,}"
    top = "all one sign" if len(set(top_signs)) == 1 else "mixed"
    return full, top


col_means = np.asarray(A_log.mean(axis=0)).ravel()           # column means of A_log (what PCA subtracts)
assert np.allclose(col_means, pca.mean_), "PCA mean_ differs from A_log column means"
mu_hat = col_means / np.linalg.norm(col_means)


def cosine(v, u):
    return float(v @ u / (np.linalg.norm(v) * np.linalg.norm(u)))


comp_tables, comp_info = {}, {}
for label, mdl in (("PCA", pca), ("Truncated SVD", svd)):
    v = mdl.components_[0]
    tbl = top_products_for_component(mdl, product_index, component=0, n=10)
    top_idx = np.argsort(-np.abs(v))[:10]
    full, top = sign_pattern(v, top_idx)
    cos_mu = cosine(v, mu_hat)
    comp_tables[label] = tbl
    comp_info[label] = {"sign_full": full, "sign_top10": top, "cos_mean": cos_mu,
                        "share_energy_or_var": (100 * mdl.explained_variance_ratio_[0]) if label == "PCA"
                        else 100 * mdl.singular_values_[0] ** 2 / energy}
    print(f"=== {label}: component 1 ===")
    print(f"sign pattern (all {m:,} weights): {full} | top-10 weights: {top}")
    print(f"cosine(component 1, normalised column means of A_log) = {cos_mu:+.4f}  (|cos| = {abs(cos_mu):.4f})")
    share_label = "explained variance" if label == "PCA" else "energy share"
    print(f"component 1 {share_label}: {comp_info[label]['share_energy_or_var']:.2f}%")
    print(tbl.to_string())
    print()

# how different are the two leading directions from each other?
print(f"|cosine(SVD component 1, PCA component 1)| = {abs(cosine(svd.components_[0], pca.components_[0])):.4f}")
print(f"cosine(normalised column means, best-seller popularity vector n_customers) = {cosine(col_means, product_index['n_customers'].to_numpy().astype(float)):.4f}")

=== PCA: component 1 ===
sign pattern (all 3,687 weights): mixed: 3,680 positive / 7 negative of 3,687 | top-10 weights: all one sign
cosine(component 1, normalised column means of A_log) = +0.9660  (|cos| = 0.9660)
component 1 explained variance: 9.30%
     StockCode                         Description    weight  n_customers_train
rank                                                                           
1        21212    PACK OF 72 RETRO SPOT CAKE CASES  0.122841               1058
2       85099B             JUMBO BAG RED RETROSPOT  0.106264                898
3        20725             LUNCH BAG RED RETROSPOT  0.103619                805
4       85123A  WHITE HANGING HEART T-LIGHT HOLDER  0.094048               1422
5        84991         60 TEATIME FAIRY CAKE CASES  0.086937                784
6        22382          LUNCH BAG SPACEBOY DESIGN   0.082876                735
7        21977  PACK OF 60 PINK PAISLEY CAKE CASES  0.081750                722
8        22384            

## 5. Filled brief table

The "What component 1 represents, in words" row holds the author's wording (written after inspecting section 4).

In [7]:
def descs(tbl):
    return "; ".join(tbl["Description"].str.strip().tolist())


def pct(x):
    return f"{x:.2f}%"


rows = [
    ("Estimator", f"PCA(n_components={K}, svd_solver={pca_solver!r}, random_state={SEED})",
                  f"TruncatedSVD(n_components={K}, algorithm='randomized', n_iter={N_ITER}, random_state={SEED})"),
    ("Input", "same sparse A_log (5,471 x 3,687); columns centred implicitly by sklearn", "sparse A_log as stored (no centring)"),
    ("Solver actually used", pca_solver, "randomized"),
    ("Fit time (s), median of 3", f"{np.median(pca_secs):.3f}", f"{np.median(svd_secs):.3f}"),
    ("Peak memory during fit (MB, tracemalloc)", f"{pca_peak:.1f}", f"{svd_peak:.1f}"),
    ("Stored factors (MB, float64: customer scores + product components)", f"{factors_mb:.2f} (+{pca_mean_mb:.2f} for mean_)", f"{factors_mb:.2f}"),
    ("Relative reconstruction error sqrt(1 - sum(sigma^2)/||A||_F^2)", "n/a (centred model)", f"{svd_rel_err:.4f}"),
    ("Energy kept (% of ||A||_F^2)", "n/a", pct(svd_energy_pct)),
    ("Explained variance (% of centred variance)", pct(pca_evr_pct), "n/a"),
    ("hit@10 (1,931 eval customers)", pct(quality["PCA"]["hit@10"]), pct(quality["Truncated SVD"]["hit@10"])),
    ("recall@10", pct(quality["PCA"]["recall@10"]), pct(quality["Truncated SVD"]["recall@10"])),
    ("precision@10", pct(quality["PCA"]["precision@10"]), pct(quality["Truncated SVD"]["precision@10"])),
    ("Best-seller baseline hit@10 (same for both)", pct(base_row["hit@10"]), pct(base_row["hit@10"])),
    ("Scoring note", "customer rows NOT centred at scoring time (raw A_log rows @ P @ P.T)", "raw A_log rows @ P @ P.T"),
    ("Component 1: top 10 products by absolute weight", descs(comp_tables["PCA"]), descs(comp_tables["Truncated SVD"])),
    ("Component 1: sign pattern", comp_info["PCA"]["sign_full"], comp_info["Truncated SVD"]["sign_full"]),
    ("Component 1: share of variance / energy", pct(comp_info["PCA"]["share_energy_or_var"]), pct(comp_info["Truncated SVD"]["share_energy_or_var"])),
    ("cosine(component 1, normalised column means of A_log)", f"{comp_info['PCA']['cos_mean']:+.4f}", f"{comp_info['Truncated SVD']['cos_mean']:+.4f}"),
    ("What component 1 represents, in words",
     "Nearly the same direction (0.996 to SVD's). After centring it measures how much a customer differs from the average in volume, so heavy buyers sit at one end and light buyers at the other. Seven products have small negative weights.",
     "Overall buying volume. It points almost exactly along the average customer's basket (cosine 0.99) and its top products are the best sellers. Every weight is positive, so customers who buy more of the popular range score higher."),
]
table = pd.DataFrame(rows, columns=["Measure", "PCA", "Truncated SVD"])

csv_path = REPORTS / "ml5_comparison.csv"
md_path = REPORTS / "ml5_comparison.md"
table.to_csv(csv_path, index=False)

lines = ["# ML5 - PCA vs Truncated SVD (k = 50) on the frozen A_log matrix", "",
         "| Measure | PCA | Truncated SVD |", "|---|---|---|"]
for _, r in table.iterrows():
    lines.append(f"| {r['Measure']} | {r['PCA']} | {r['Truncated SVD']} |")
lines += ["", "Notes:",
          f"- Fit time is the median of 3 fresh fits; peak memory is from one extra fit with `tracemalloc` "
          f"(Python-allocator view; ARPACK/LAPACK workspaces are not counted).",
          "- Both methods are scored with the identical pipeline from `src/recommend.py`: P = components_.T with unit-norm rows, "
          "scores = A_log[row] @ P @ P.T, already-bought products excluded, top 10. PCA rows are NOT centred at scoring time.",
          "- Component signs are arbitrary; 'all one sign' is the pattern that matters, not positive vs negative.",
          "- Source notebook: `notebooks/08_pca_vs_svd.ipynb`."]
md_path.write_text("\n".join(lines) + "\n")

print(table.to_string(index=False, max_colwidth=70, justify="left"))
print(f"\nsaved -> {csv_path.relative_to(ROOT)}")
print(f"saved -> {md_path.relative_to(ROOT)}")

Measure                                                            PCA                                                                    Truncated SVD                                                         
                                                         Estimator             PCA(n_components=50, svd_solver='arpack', random_state=42) TruncatedSVD(n_components=50, algorithm='randomized', n_iter=7, ran...
                                                             Input same sparse A_log (5,471 x 3,687); columns centred implicitly by sk...                                   sparse A_log as stored (no centring)
                                              Solver actually used                                                                 arpack                                                             randomized
                                         Fit time (s), median of 3                                                                  0.095                           

## 6. Why PCA and SVD agree here

Centring subtracts each column's mean. On a 97.95%-sparse matrix those means are tiny next to the stored values, so the centred and uncentred problems are nearly the same.

In [8]:
col_means = np.asarray(A_log.mean(axis=0)).ravel()
mean_col_mean = float(col_means.mean())
max_col_mean = float(col_means.max())
mean_nonzero = float(A_log.data.mean())
ratio_pct = 100 * mean_col_mean / mean_nonzero
sparsity_pct = 100 * (1 - A_log.nnz / (n * m))

print(f"sparsity                                   : {sparsity_pct:.2f}% zeros")
print(f"mean of all column means                   : {mean_col_mean:.4f}")
print(f"max column mean                            : {max_col_mean:.4f}")
print(f"mean non-zero value                        : {mean_nonzero:.4f}")
print(f"mean column mean as % of mean non-zero     : {ratio_pct:.2f}%")

why = [
    "", "## Why PCA and SVD agree here", "",
    "| Statistic of A_log | Value |", "|---|---|",
    f"| Mean of all column means | {mean_col_mean:.4f} |",
    f"| Max column mean | {max_col_mean:.4f} |",
    f"| Mean non-zero value | {mean_nonzero:.4f} |",
    f"| Mean column mean as % of mean non-zero value | {ratio_pct:.2f}% |",
    "",
    "Because A_log is 97.95% zeros, each column mean is small relative to the non-zero values, so centring shifts the data "
    "very little and PCA finds almost the same directions as SVD (component 1 cosine 0.996). The ML1 concern about centring "
    "is correct in principle but has a small effect on this matrix.",
]
with md_path.open("a") as fh:
    fh.write("\n".join(why) + "\n")
print(f"appended section -> {md_path.relative_to(ROOT)}")

sparsity                                   : 97.95% zeros
mean of all column means                   : 0.0436
max column mean                            : 0.7382
mean non-zero value                        : 2.1271
mean column mean as % of mean non-zero     : 2.05%
appended section -> reports/ml5_comparison.md


## Stop here

The table is for the memo's "PCA vs SVD" section. The method choice is recorded in `decision_log.md` (ML5 outcome), not here.